# EA3 - Actividad 3.1b: Ingesta de Datos en Tiempo Real

## Objetivos
- Usar el generador de datos de streaming para simular flujos de datos
- Explorar y analizar datos generados en formato JSONL
- Enviar datos generados a Kafka para procesamiento posterior

> **IMPORTANTE:** Este notebook requiere el perfil **completo** de Docker.
> ```bash
> docker-compose --profile completo up -d
> ```
> Espera ~1 minuto para que Kafka y los demas servicios inicien completamente.

## Conceptos Clave

### Streams de Datos

Un **stream** (flujo) de datos es una secuencia continua e ilimitada de registros que llegan a lo largo del tiempo. A diferencia del procesamiento batch (por lotes), donde los datos son finitos y estaticos, en streaming los datos llegan continuamente.

### Simulacion de Streams

Para aprender sobre procesamiento en tiempo real sin depender de sistemas productivos, usamos **generadores de datos** que simulan diferentes tipos de eventos:

| Tipo de Evento | Descripcion | Campos Principales |
|----------------|-------------|--------------------|
| **Transacciones** | Ventas en tiendas | id, monto, producto, tienda, metodo_pago |
| **Logs** | Registros de servidor web | endpoint, status_code, response_time, ip |
| **IoT** | Lecturas de sensores | sensor_id, temperatura, humedad, ubicacion |

### Formato JSONL (JSON Lines)

JSONL es un formato donde **cada linea es un documento JSON independiente**. Es ideal para streaming porque:
- Se puede leer linea por linea (no necesita cargar todo en memoria)
- Es facil de generar incrementalmente (append)
- Spark lo lee nativamente

```
{"id": "TX-001", "monto": 15000, "producto": "Laptop"}
{"id": "TX-002", "monto": 8500, "producto": "Tablet"}
{"id": "TX-003", "monto": 3200, "producto": "Teclado"}
```

## Setup

In [ ]:
import json
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ingesta_rt") \
    .master("local[*]") \
    .getOrCreate()

# Crear directorio de salida si no existe
os.makedirs("/home/jovyan/datos/streaming", exist_ok=True)

print(f"Spark version: {spark.version}")
print("Setup completado.")

## 1. Generar Datos de Transacciones

Usamos el script `generar_datos_streaming.py` para crear un archivo JSONL con transacciones de ejemplo.

In [ ]:
# Generar 500 transacciones de ejemplo a archivo
%run /home/jovyan/scripts/generar_datos_streaming.py --tipo transacciones --archivo /home/jovyan/datos/streaming/transacciones_sample.jsonl --cantidad 500

## 2. Leer JSONL con Spark

Spark puede leer archivos JSONL directamente con `spark.read.json()`. Cada linea se convierte en una fila del DataFrame.

In [ ]:
df_tx = spark.read.json("/home/jovyan/datos/streaming/transacciones_sample.jsonl")
df_tx.show(5)
df_tx.printSchema()

In [ ]:
print(f"Total de transacciones: {df_tx.count()}")
print(f"Columnas: {df_tx.columns}")

## 3. Explorar los Datos Generados

Veamos las estadisticas y la distribucion de los datos de transacciones.

In [ ]:
# Estadisticas descriptivas
df_tx.describe().show()

In [ ]:
# Distribucion por producto
print("Transacciones por producto:")
df_tx.groupBy("producto").count().orderBy("count", ascending=False).show()

In [ ]:
# Monto promedio por tienda
print("Monto promedio por tienda:")
df_tx.groupBy("tienda_id").agg(
    F.avg("monto").alias("monto_promedio"),
    F.count("*").alias("num_transacciones")
).orderBy("tienda_id").show()

## 4. Generar Datos de Logs

Ahora generamos datos de tipo **logs**, que simulan registros de un servidor web.

In [ ]:
# Generar 500 registros de logs
%run /home/jovyan/scripts/generar_datos_streaming.py --tipo logs --archivo /home/jovyan/datos/streaming/logs_sample.jsonl --cantidad 500

In [ ]:
# Leer logs con Spark
df_logs = spark.read.json("/home/jovyan/datos/streaming/logs_sample.jsonl")
df_logs.show(5)
df_logs.printSchema()

In [ ]:
# Explorar los logs
print(f"Total de logs: {df_logs.count()}")

print("\nDistribucion por status_code:")
df_logs.groupBy("status_code").count().orderBy("count", ascending=False).show()

print("Response time promedio por endpoint:")
df_logs.groupBy("endpoint").agg(
    F.avg("response_time").alias("response_time_promedio")
).orderBy("response_time_promedio", ascending=False).show()

## 5. Generar Datos IoT

Finalmente, generamos datos de tipo **IoT** que simulan lecturas de sensores de temperatura y humedad.

In [ ]:
# Generar 500 lecturas de sensores IoT
%run /home/jovyan/scripts/generar_datos_streaming.py --tipo iot --archivo /home/jovyan/datos/streaming/iot_sample.jsonl --cantidad 500

In [ ]:
# Leer datos IoT con Spark
df_iot = spark.read.json("/home/jovyan/datos/streaming/iot_sample.jsonl")
df_iot.show(5)
df_iot.printSchema()

In [ ]:
# Explorar datos IoT
print(f"Total de lecturas IoT: {df_iot.count()}")

print("\nEstadisticas de temperatura:")
df_iot.select("temperatura").describe().show()

print("Temperatura promedio por ubicacion:")
df_iot.groupBy("ubicacion").agg(
    F.avg("temperatura").alias("temp_promedio"),
    F.avg("humedad").alias("humedad_promedio")
).show()

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Analisis de logs
# =============================================================
# Generamos 1000 logs, los leemos con Spark y analizamos

# 1. Generar datos
%run /home/jovyan/scripts/generar_datos_streaming.py --tipo logs --archivo /home/jovyan/datos/streaming/logs_ejercicio.jsonl --cantidad 1000

> **Nota docente:** El generador crea datos aleatorios pero realistas. Los estudiantes deben entender que en produccion estos datos vendrian de un servidor web real (nginx, Apache, etc.).

In [ ]:
# 2. Leer el archivo JSONL con Spark
df_logs = spark.read.json("/home/jovyan/datos/streaming/logs_ejercicio.jsonl")
print(f"Total de logs generados: {df_logs.count()}")
df_logs.show(5)

# 3. Distribucion de status_code
print("\nDistribucion de status_code:")
df_logs.groupBy("status_code").count().orderBy("status_code").show()

# 4. Response time promedio por endpoint
print("Response time promedio por endpoint:")
df_logs.groupBy("endpoint").agg(
    F.avg("response_time").alias("avg_response_ms")
).orderBy("avg_response_ms", ascending=False).show()

> **Nota docente:** Este ejercicio practica el patron completo: generar -> leer -> analizar. La distribucion de `status_code` revela la salud del servidor (mayoria 200 = bien, muchos 500 = problemas). El response_time por endpoint permite identificar endpoints lentos que necesitan optimizacion. Error comun: confundir `response_time` con `response_time_ms` segun lo que genere el script.

In [ ]:
# =============================================================
# EJERCICIO 2: Deteccion de anomalias IoT
# =============================================================
# Generamos datos IoT y detectamos sensores fuera de rango

# 1. Generar datos
%run /home/jovyan/scripts/generar_datos_streaming.py --tipo iot --archivo /home/jovyan/datos/streaming/iot_ejercicio.jsonl --cantidad 1000

In [ ]:
# 2. Leer el archivo con Spark
df_iot = spark.read.json("/home/jovyan/datos/streaming/iot_ejercicio.jsonl")
print(f"Total lecturas IoT: {df_iot.count()}")

# 3. Encontrar sensores fuera de rango (temp < 10 o temp > 30)
df_anomalias = df_iot.filter(
    (F.col("temperatura") < 10) | (F.col("temperatura") > 30)
)

total = df_iot.count()
anomalas = df_anomalias.count()
print(f"\nLecturas fuera de rango: {anomalas} de {total} ({anomalas/total*100:.1f}%)")

# 4. Mostrar lecturas anomalas
print("\nLecturas anomalas:")
df_anomalias.show(10)

# Anomalias por sensor
print("Anomalias por sensor:")
df_anomalias.groupBy("sensor_id").count().orderBy("count", ascending=False).show()

> **Nota docente:** La deteccion de anomalias es un caso de uso clasico en IoT. Los umbrales (10 y 30 grados) son arbitrarios para el ejercicio; en produccion se definirian segun el contexto (ej: un datacenter tiene umbrales mas estrictos que un invernadero). Error comun: usar `&` en lugar de `|` (AND en vez de OR), lo cual buscaria temperaturas simultaneamente menores a 10 Y mayores a 30 (imposible). Recordar que en PySpark las condiciones deben ir entre parentesis cuando se combinan con `|` o `&`.

---
## Resumen

En esta actividad aprendimos:

1. **Streams de datos:** Flujos continuos de eventos que llegan a lo largo del tiempo
2. **Simulacion:** Uso de generadores para crear datos de prueba realistas
3. **Formato JSONL:** Un JSON por linea, ideal para streaming y lectura incremental
4. **Tipos de eventos:** Transacciones, logs de servidor y lecturas IoT
5. **Lectura con Spark:** `spark.read.json()` lee JSONL nativamente
6. **Analisis exploratorio:** Distribucion, estadisticas y deteccion de anomalias

---
## Desafio Extra (Opcional)

**Enviar datos generados a Kafka:**

Usa el modo Kafka del generador para enviar datos directamente a un topic de Kafka, y luego consumelos desde otra celda para verificar que llegaron correctamente.

In [ ]:
# =============================================================
# DESAFIO: Enviar datos generados a Kafka
# =============================================================
# Usamos el generador en modo Kafka

# 1. Generar 100 transacciones y enviar directamente a Kafka
%run /home/jovyan/scripts/generar_datos_streaming.py --tipo transacciones --cantidad 100 --topic ingesta-desafio

print("100 transacciones enviadas al topic 'ingesta-desafio'")

> **Nota docente:** El modo Kafka del generador abstrae la creacion del producer y el envio. En produccion, los datos fluyen automaticamente desde los sistemas fuente (POS, sensores, servidores web) hacia Kafka sin intervencion manual.

In [ ]:
# =============================================================
# DESAFIO (continuacion): Consumir mensajes de Kafka
# =============================================================
from kafka import KafkaConsumer

KAFKA_BROKER = "kafka:29092"

consumer = KafkaConsumer(
    "ingesta-desafio",
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=10000
)

import json
count = 0
for msg in consumer:
    count += 1
    if count <= 5:
        print(f"Mensaje {count}: {msg.value}")

consumer.close()
print(f"\nTotal mensajes recibidos: {count}")
print(f"Verificacion: {'OK' if count == 100 else f'Se esperaban 100, se recibieron {count}'}") 

> **Nota docente:** Solo mostramos los primeros 5 mensajes para no saturar la salida, pero contamos todos. Este patron (consumir todo, mostrar muestra) es comun en debugging de pipelines. La verificacion final cierra el ciclo: generar -> enviar a Kafka -> consumir -> validar integridad.

In [ ]:
# Detener SparkSession
spark.stop()
print("SparkSession detenida correctamente.")